In [1]:
import pandas as pd
from pathlib import Path
import re
from difflib import SequenceMatcher
from unidecode import unidecode
from scipy.optimize import linear_sum_assignment
import numpy as np
from rapidfuzz import fuzz


MAIN_FOLDER = r"C:\Users\L14\Downloads\ABH_CSV"



def normalize_text(x):
    if pd.isna(x):
        return ""
    return unidecode(str(x)).lower().strip()

def clean_name(x):
    return normalize_text(x).split("(")[0].split(":")[-1].strip()

def make_code_vois(village, name):
    return normalize_text(village) + "-" + clean_name(name)

def fallback(main, backup):
    return main.replace("", pd.NA).fillna(backup)

#path = r"C:\Users\L14\Documents\SITUATION GLOBALE\CONTROLE_02_02_2026\CONTROLE CTB REMARQUES_total_fusionne.xlsx"

path_maj_gestionnaire = fr"{MAIN_FOLDER}\DEMANDES_total_fusionne.xlsx"
path_voisins_pvcl = fr"{MAIN_FOLDER}\VOISINS_PVCL_total_fusionne.xlsx"
path_voisins_signes = fr"{MAIN_FOLDER}\VOISINS_PRESENCE_total_fusionne.xlsx"
path_voisins_non_par = fr"{MAIN_FOLDER}\VOISINS_NON_PAR_total_fusionne.xlsx"
path_etat_voisin = fr"{MAIN_FOLDER}\Etat_Tonkpi_Liv1_21-03-26.xlsx"
path_etat_voisin_csv = fr"{MAIN_FOLDER}\data_tokpi_19_03_26.csv"
path_voisins_signes_cni = r"C:\Users\L14\Desktop\scripts_python\matches_repr_test.csv"
path_chef_village = r"C:\Users\L14\Desktop\controle_qualite\chef_de_village_avec_cni.csv"

paths_rem = Path(fr"{MAIN_FOLDER}\ALL_CSV\CONTROLE CTB REMARQUES")
paths_dig = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG")
paths_ctb = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB")

out_add_to_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\ADD_TO_CTB")
out_no_mask_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\NO_MASK")
out_replace_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\REPLACE_CTB")
out_dig_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG")
out_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB")
out_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\UPDATES_FILES")
out_to_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB\ADD_NEIGHBOR\TO_CTB")
out_ctb_manual_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB\ADD_NEIGHBOR\MANUAL")

paths_action_dig = paths_dig.glob("*.csv")
paths_action_ctb = paths_ctb.glob("*.csv")
CSVs = paths_dig.glob("*.csv")
files_rem = paths_rem.rglob("*.xlsx")

df_maj_gestionnaire = pd.read_excel(path_maj_gestionnaire, engine='openpyxl')
df_maj_gestionnaire["applicantNumber"] = df_maj_gestionnaire["applicantNumber"].astype(str).str.replace("APT", "", regex=False).str.strip()
etat_voisins = pd.read_excel(path_etat_voisin, engine='openpyxl')
#etat_voisins = pd.read_csv(path_etat_voisin_csv)
df_voisins_signes = pd.read_excel(path_voisins_signes, engine='openpyxl')
df_voisins_non_par = pd.read_excel(path_voisins_non_par, engine='openpyxl')
df_voisins_signes_ctrl = pd.read_excel(path_voisins_signes, engine='openpyxl')
df_voisins_pvcl = pd.read_excel(path_voisins_pvcl, engine='openpyxl')
#df_voisins_signes_cni = pd.read_csv(path_voisins_signes_cni, sep=";", encoding="cp1252")
df_voisins_signes_cni = pd.read_csv(path_voisins_signes_cni, sep=";", encoding="utf-8-sig")
df_chef_village = pd.read_csv(path_chef_village)

#controle_voisins = pd.read_excel(path, engine='openpyxl')

df_maj_gestionnaire.loc[:, "village"] = (
    df_maj_gestionnaire["village"]
      .apply(lambda x: unidecode(x) if pd.notna(x) else x)
      .str.lower()
      .str.strip()
    )

df_maj_gestionnaire.loc[:, "code_par"] = df_maj_gestionnaire.loc[:, "village"] + "-" + df_maj_gestionnaire.loc[:, "numParcelleOF_c"]

#df_maj_gestionnaire.set_index("code_par", inplace=True)

etat_voisins["code_voisin_unique"] = (etat_voisins["indice_unique_voisin"].astype(str).str.strip() + "__" + etat_voisins["num_demande"].astype(str).str.strip())
etat_voisins = etat_voisins[etat_voisins["num_demande"].notna()]
etat_voisins_duplicated = etat_voisins[etat_voisins["code_voisin_unique"].duplicated()]
etat_voisins = etat_voisins.drop_duplicates(subset="code_voisin_unique")
etat_voisins['cod_sp'] = etat_voisins['num_demande'].astype(str).str.split("-").str[0]

for df in [df_voisins_signes_cni, df_voisins_pvcl, df_voisins_non_par]:
    df["village"] = df["village"].apply(normalize_text)

    df_voisins_signes_cni["nom_sig"] = df_voisins_signes_cni["nom_sig"].apply(clean_name)
    df_voisins_pvcl["nameOfNeighbor"] = df_voisins_pvcl["nameOfNeighbor"].apply(clean_name)
    df_voisins_non_par["nameOfPerson_control"] = df_voisins_non_par["nameOfPerson_control"].apply(clean_name)

    df_voisins_signes_cni["code_vois"] = make_code_vois(
        df_voisins_signes_cni["village"],
        df_voisins_signes_cni["nom_sig"]
    )

    df_voisins_pvcl["code_vois"] = make_code_vois(
        df_voisins_pvcl["village"],
        df_voisins_pvcl["nameOfNeighbor"]
    )

    df_voisins_non_par["code_vois"] = make_code_vois(
        df_voisins_non_par["village"],
        df_voisins_non_par["nameOfPerson_control"]
    )

df_voisins_signes_cni.set_index("code_vois", inplace=True)
df_voisins_pvcl.set_index("code_vois", inplace=True)
df_voisins_non_par.set_index("code_vois", inplace=True)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
### Generate the file actions_digifor and actions_ctb to update data

#print(etat_voisins_duplicated.shape)
#print(etat_voisins_duplicated["code_voisin_unique"].head(10))

assert etat_voisins["code_voisin_unique"].is_unique, "❌ Le code voisin unique ne l'est pas"
etat_voisins.set_index("code_voisin_unique", drop=False, inplace=True)

def nearest_text_match(target, candidates, target_type='dig', min_score=0.5):
        """
        target : str (voisin DIGIFOR)
        candidates : list[str] (voisins CTB)
        """
        best = None
        unmatched = set(candidates)
        best_score = 0

        for c in candidates:
            if ("Aucun voisin" in target) or ("Aucun voisin" in c):
                    score = text_similarity(target, c)
            elif target_type == "dig":
                #print(c)
                score = text_similarity(target.split('(')[0], c.split(':')[-1].split('(')[0])
            elif target_type == "ctb":
                score = text_similarity(target.split(':')[-1].split('(')[0], c.split('(')[0])
            else:
                score = text_similarity(target, c)

            if score > best_score:
                best = c
                best_score = score
            
        if best_score >= min_score:
            unmatched -= {best}
            return best, unmatched, best_score

        return None, unmatched, best_score

def best_match_nn(digs, ctbs, min_score=0.6):
    if not digs or not ctbs:
        return [], digs.copy(), ctbs.copy()

    matrix = np.zeros((len(digs), len(ctbs)))

    for i, d in enumerate(digs):
        for j, c in enumerate(ctbs):
            if ("Aucun voisin" in d) or ("Aucun voisin" in c):
                score = text_similarity(d, c)
            else:
                score = text_similarity(d.split('(')[0], c.split(':')[-1].split('(')[0])
            matrix[i, j] = score

    row_ind, col_ind = linear_sum_assignment(-matrix)

    matches = []
    used_dig = set()
    used_ctb = set()

    for i, j in zip(row_ind, col_ind):
        score = matrix[i, j]
        if score >= min_score:
            matches.append((digs[i], ctbs[j], score))
            used_dig.add(digs[i])
            used_ctb.add(ctbs[j])

    unmatched_dig = [d for d in digs if d not in used_dig]
    unmatched_ctb = [c for c in ctbs if c not in used_ctb]

    return matches, unmatched_dig, unmatched_ctb

def split_clean(value):
    if pd.isna(value) or value == "RAS":
        return []
    return [v.strip() for v in value.split(",") if v.strip()]

def is_par(v): return re.match(r"^par\d+", v)
def is_riv(v): return re.match(r"^riv\d+", v)
def is_tit(v): return re.match(r"^tit\d+", v)
def is_zone(v): return re.match(r"^zone\d+", v)
def is_det(v): return "det_poly" in v
def is_signe(v): return "(signe)" in v or "(repr)" in v or "(non_par)" in v

def text_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def get_voisin_info(v, row_code):
    if 'par' in v:
        code_v = v.split(':')[0] + "__" + row_code
        if code_v in etat_voisins.index:
            r = etat_voisins.loc[code_v]
            code_voisin = r["num_plle_voisin"]
            village = r["cod_sp"]
            num_voisin = r["voisin_num_demande"]
            return num_voisin, code_voisin, r["type_voisin"], r["orientation"]
    return None, None, None, None

def compare_names(row, code):
    
    rows_dig = []
    rows_ctb = []

    voisins_dig = split_clean(row["A corriger sur CTB ou Supprimer sur DIGIFOR"])
    voisins_ctb = split_clean(row["A ajouter ou Corriger sur DIGIFOR"])

    taille_dig = len(voisins_dig)
    taille_ctb = len(voisins_ctb)

    def add_row(old, new,code_voisin, code_voisin_plle, type_voisin, position, action, comment):
        rows_dig.append({
            "code": row["code"],
            "code_parcelle": row["numParcelleOF"],
            "village": row["village"],
            "date": row["requestDate"],
            "ancien_nom": old,
            "nouveau_nom": new,
            "code_voisin": code_voisin,
            "code_voisin_plle": code_voisin_plle,
            "type_voisin": type_voisin,
            "position": position,
            "action": action,
            "commentaire": comment
        })
    
    def add_row_ctb(old, new, type_voisin, position, action, comment):
        rows_ctb.append({
            "code": row["code"],
            "code_parcelle": row["numParcelleOF"],
            "village": row["village"],
            "date": row["requestDate"],
            "ancien_nom": old,
            "nouveau_nom": new,
            "type_voisin": type_voisin, 
            "position": position,
            "action": action,
            "commentaire": comment
        })

    
    if voisins_dig == ["Aucun voisin dans le pvcl presence"] or voisins_ctb == ["Aucun voisin dans le ctb"]:
        return [], []
    
    # 1️⃣ DIGIFOR RAS
    elif not voisins_dig and voisins_ctb:
        for v in voisins_ctb:
            code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout depuis CTB (DIGIFOR = RAS)")

    # 2️⃣ CTB RAS
    elif not voisins_ctb and voisins_dig:
        for v in voisins_dig:
            add_row(v, None, "", "", "", "", "DELETE", "Suppression (CTB = RAS)")

    # 3️⃣ 1–1
    elif taille_ctb == taille_dig == 1:
        ctb, dig = voisins_ctb[0], voisins_dig[0]
        code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(ctb, row["code"])
        add_row(None, ctb, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajouter le voisin ctb = (par,zone) 1–1")
        add_row(dig, None, "", "", "", "", "DELETE", "Supprimer le voisin dig, ctb = (par,zone) 1–1")

    # 4️⃣ 1 DIG vs N CTB
    elif taille_dig == 1:
        dig = voisins_dig[0]

        for v in voisins_ctb:
            code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout de voisin all(voisin par,,zone)")
        add_row(dig, None, "", "", "", "", "DELETE", "Suppression ancien voisin (all ctb =par,,zone)")


    # 5️⃣ 1 CTB vs N DIG
    elif taille_ctb == 1:
        ctb = voisins_ctb[0]
        code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(ctb, row["code"])
        add_row(None, ctb, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det)")
        for v in voisins_dig:
            add_row(v, None, "", "", "", "", "DELETE", "Nettoyage DIGIFOR")

                
    # 6️⃣ N–N
    else:
        for v in voisins_ctb:
            code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det ou tit ou zone)")
        for v in voisins_dig:
            add_row(v, None, "", "", "", "", "DELETE", "Nettoyage DIGIFOR all(voisin ctb = par ou det ou tit ou zone)")

    return rows_dig, rows_ctb

for file in files_rem:
    print(file)
    controle_voisins = pd.read_excel(file, engine='openpyxl')
    all_rows_dig = []
    all_rows_ctb = []
    
    for idx, row in controle_voisins.iterrows():
        result_dig, result_ctb = compare_names(row, code=idx)
        all_rows_dig.extend(result_dig)
        all_rows_ctb.extend(result_ctb)

    result_df_dig = pd.DataFrame(all_rows_dig,
                            columns=["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "type_voisin", "code_voisin", "code_voisin_plle", "position", "action", "commentaire"])
    #result_df_dig["id"] = result_df_dig.index
    result_df_ctb = pd.DataFrame(all_rows_ctb,
                            columns=["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "type_voisin", "position", "action", "commentaire"])
    #result_df_ctb["id"] = result_df_ctb.index
    result_df_dig.to_csv(f"{out_dig_folder}/{file.name.split('.')[0]}_actions_digifor.csv",
                    index=False,
                    sep=";",
                    encoding="utf-8-sig")

In [ ]:
def fuzzy_match(row, ref_df, threshold=85):
    if pd.notna(row["cni_voisin"]):
        return row["cni_voisin"], row["signature_voisin"], row["name_sig_voisin"], "exact"

    village = row["village"]
    name = clean_name(row["name"])

    candidates = ref_df[ref_df["village"] == village]

    best_score = 0
    best_row = None

    for _, r in candidates.iterrows():
        score = fuzz.token_set_ratio(name, r["nom_sig"])
        score_repr = fuzz.token_set_ratio(name, str(r.get("nom_repr", "")))

        score_final = max(score, score_repr)

        if score_final > best_score:
            best_score = score_final
            best_row = r

    if best_score >= threshold and best_row is not None:
        return (
            best_row["number_cni"],
            best_row["signatory_photo"],
            best_row["nom_sig_original"],
            "fuzzy"
        )

    return row["cni_voisin"], row["signature_voisin"], row["name_sig_voisin"], "no_match"

for path_action_dig in paths_action_dig:

    print("Processing:", path_action_dig)

    df_action_dig = pd.read_csv(path_action_dig, sep=";", encoding="utf-8-sig")

    # --- NORMALISATION
    df_action_dig["village"] = df_action_dig["village"].apply(normalize_text)

    df_action_dig["name"] = (
        df_action_dig["nouveau_nom"]
        .fillna(df_action_dig["ancien_nom"])
        .apply(clean_name)
    )

    df_action_dig["code_vois"] = make_code_vois(
        df_action_dig["village"],
        df_action_dig["name"]
    )

    # ------------------------------------------------
    # MERGE EXACT
    # ------------------------------------------------
    df_action_dig = df_action_dig.join(
        df_voisins_signes_cni[
            ["nom_sig_original", "nom_repr", "code_sig", "signatory_photo", "number_cni"]
        ],
        on="code_vois"
    )

    df_action_dig.rename(columns={
        "nom_sig_original": "name_sig_voisin",
        "number_cni": "cni_voisin",
        "signatory_photo": "signature_voisin"
    }, inplace=True)

    # ------------------------------------------------
    # FUZZY FALLBACK 🔥
    # ------------------------------------------------
    mask_missing = df_action_dig["cni_voisin"].isna()

    df_action_dig.loc[mask_missing, [
        "cni_voisin",
        "signature_voisin",
        "name_sig_voisin",
        "match_type"
    ]] = df_action_dig[mask_missing].apply(
        lambda row: fuzzy_match(row, df_voisins_signes_cni.reset_index()),
        axis=1,
        result_type="expand"
    )

    df_action_dig["match_type"] = df_action_dig["match_type"].fillna("exact")

    # ------------------------------------------------
    # FALLBACK GESTIONNAIRE
    # ------------------------------------------------
    df_action_dig["code_voisin"] = df_action_dig["code_voisin"].astype(str)

    df_action_dig = df_action_dig.merge(
        df_maj_gestionnaire[
            ["applicantNumber", "identityDocumentNumber_x", "identityDocumentPhoto"]
        ],
        left_on="code_voisin",
        right_on="applicantNumber",
        how="left"
    )

    df_action_dig["cni_voisin"] = fallback(
        df_action_dig["cni_voisin"],
        df_action_dig["identityDocumentNumber_x"]
    )

    df_action_dig["signature_voisin"] = fallback(
        df_action_dig["signature_voisin"],
        df_action_dig["identityDocumentPhoto"]
    )

    # ------------------------------------------------
    # DEBUG 📊
    # ------------------------------------------------
    print("Total:", len(df_action_dig))
    print("CNI trouvées:", df_action_dig["cni_voisin"].notna().mean())
    print("Exact:", (df_action_dig["match_type"] == "exact").mean())
    print("Fuzzy:", (df_action_dig["match_type"] == "fuzzy").mean())

    # ------------------------------------------------
    # EXPORT
    # ------------------------------------------------
    df_action_dig.to_csv(path_action_dig, sep=";", encoding="utf-8-sig", index=False)

In [ ]:
### Generate the files add_data.csv, replace_data.csv, delete_data.csv from actions_digifor.csv
update_files_resume = []

for csv_path in CSVs:
    agent = csv_path.stem
    df_action_dig = pd.read_csv(csv_path, sep=";")
    df_action_dig.index = range(len(df_action_dig))
    
    print("Processing:", path_action_dig)

    ### Generate add_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_ajouter, type_voisin, num_voisin, cni_voisin, signature_voisin, a_signe)
    #   print(df_action_dig.columns)
    is_add = df_action_dig["action"].isin(["ADD", "ADD_SEARCH"])
    is_voisin_plle = df_action_dig["type_voisin"].isin(["voisin_plle", "limit_naturelle"])
    is_non_par = df_action_dig["nameCVGFR"].fillna("").str.strip() != ""
    is_voisin_riv = (
        (df_action_dig["type_voisin"] == "voisin_riv") &
        (df_action_dig["cni_voisin"].fillna("") != "")
    )

    mask_voisin_plle = (
        (is_add & is_voisin_plle) |
        (is_add & is_voisin_riv)
    )

    df_action_dig.loc[mask_voisin_plle, "nom_a_ajouter"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(":").str[1].str.replace("_"," ").str.strip()
    )
    df_action_dig.loc[mask_voisin_plle, "code_uniq"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(":").str[0].str.strip()
    )
    
    df_action_dig.loc[mask_voisin_plle, ["code_dig", "code_parcelle", "village", "date", "code_uniq", "nom_a_ajouter", "name_sig_voisin", "nom_repr", "nameOfNeighborOriginal", "type_voisin", "code_voisin", "num_voisin", "cni_voisin", "position_dig", "signature_voisin"]].to_csv(f"{out_folder}/{csv_path.stem}_add_data.csv", sep=",", encoding="utf-8-sig")

    ### Generate replace_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_remplacer, nom_a_ajouter, type_voisin, num_voisin, cni_voisin, signature_voisin, a_signe)

    mask_replace_d = df_action_dig["action"] == "REPLACE_D"

    df_action_dig.loc[mask_replace_d, "nom_a_ajouter"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(':', n=1).str[-1].str.replace("_"," ").str.strip()
    )
    df_action_dig.loc[mask_replace_d, "nom_a_remplacer"] = (
        df_action_dig["ancien_nom"].astype(str).str.split("(").str[0].str.replace("_"," ").str.strip()
    )
    df_action_dig.loc[mask_replace_d, "code_uniq"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(":").str[0].str.strip()
    )

    df_action_dig.loc[mask_replace_d, ["code_dig", "code_parcelle", "village", "date", "code_uniq", "nom_a_remplacer", "name_sig_voisin", "nom_repr", "nameOfNeighborOriginal", "nom_a_ajouter", "type_voisin", "num_voisin", "cni_voisin", "position_dig", "signature_voisin"]].to_csv(f"{out_folder}/{csv_path.stem}_replace_data.csv", sep=",", encoding="utf-8-sig")

    ### Generate delete_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_supprimer, type_voisin, a_signe)

    mask_delete = df_action_dig["action"] == "DELETE"

    df_action_dig.loc[mask_delete, "nom_a_supprimer"] = (
        df_action_dig["ancien_nom"].astype(str).str.split("(").str[0].str.replace("_"," ")
    )

    df_action_dig.loc[mask_delete, ["code_dig", "code_parcelle", "village", "date", "nom_a_supprimer","name_sig_voisin", "nom_repr", "nameOfNeighborOriginal", "cni_voisin", "position_dig", "type_voisin"]].to_csv(f"{out_folder}/{csv_path.stem}_delete_data.csv", sep=",", encoding="utf-8-sig")

    ### Generate non_par_data.csv ("code", "nameVoiz", "nameCVGFR", "cod_vil","firstDate","secondDate","comityAttestSignatoryCE","nameCVGFR","limitVoiz","nameVoiz","comityAttestSignatoryCVGFR","comityAttestSignatoryPCVGFR","creationDate","active")

    df_action_dig.loc[is_non_par, ["code_dig", "nameVoiz", "nameCVGFR", "firstDate", "secondDate", "comityAttestSignatoryCE","nameCVGFR","limitVoiz","nameVoiz","comityAttestSignatoryCVGFR","comityAttestSignatoryPCVGFR","creationDate","active"]].to_csv(f"{out_folder}/{csv_path.stem}_non_par_data.csv", sep=";", encoding="utf-8-sig")

    
    # ------------------------------------------------
    # Recupération signature et CNI depuis chef de village
    # ------------------------------------------------
    df_action_dig["cod_vil"] = df_action_dig['code_dig'].str.split('-').str[:2].str.join('-')
    mask_zone = df_action_dig["type_voisin"].astype(str).str.contains("zone", case=False, na=False)
    df_action_dig_zone = df_action_dig[mask_zone]
    df_action_dig_zone = df_action_dig_zone.merge(
        df_chef_village[["cod_vil", "identityDocumentNumber",
                              "identityDocumentPhoto"]],
        left_on="cod_vil",
        right_on="cod_vil",
        how="left"
    )

    df_action_dig_zone.rename(
        columns={"identityDocumentNumber": "cni_chef_village", "identityDocumentPhoto": "signature_chef_village"},
        inplace=True
    )

    df_action_dig_zone[["code_dig", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "name_sig_voisin", "nom_repr", "nameOfNeighborOriginal", "type_voisin", "cni_chef_village", "signature_chef_village"]].to_csv(f"{out_folder}/{csv_path.stem}_zone_data.csv", sep=";", encoding="utf-8-sig")
    no_mask = ~ (mask_voisin_plle | mask_replace_d | mask_delete | is_non_par | mask_zone)
    df_action_dig.loc[no_mask, ["code_dig", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "name_sig_voisin", "nom_repr", "nameOfNeighborOriginal", "type_voisin", "action"]].to_csv(f"{out_no_mask_folder}/{csv_path.stem}_nomask_data.csv", sep=";", encoding="utf-8-sig") 

    update_files_resume.append({
        "Agent": agent,
        "Total lignes" : len(df_action_dig),
        "ADD" : mask_voisin_plle.sum(),
        "REPLACE" : mask_replace_d.sum(),
        "DELETE": mask_delete.sum(),
        "NON_PAR": is_non_par.sum(),
        "NO_MASK" : no_mask.sum(),
        "Somme totale" :
        mask_voisin_plle.sum()
        + mask_replace_d.sum()
        + mask_delete.sum()
        + is_non_par.sum()
        + no_mask.sum()
    })

df_update_files = pd.DataFrame(update_files_resume,
                               columns=["Agent", "Total lignes", "ADD", "REPLACE", "DELETE", "NON_PAR", "NO_MASK", "Somme totale"])

df_update_files.to_csv(f"resume_data.csv", sep=";", encoding="utf-8-sig")